# Colab Two-Tower Baseline Training

This notebook runs the serious baseline config on Colab using the Gold dataset already saved in Drive.

Silver data is not required for this step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Expected paths

Gold data:

```text
/content/drive/MyDrive/recsys/data/gold/two_tower/v1_ready/
```

Artifacts:

```text
/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v1/
```

In [ ]:
from pathlib import Path

GOLD_ROOT = Path('/content/drive/MyDrive/recsys/data/gold/two_tower/v1_ready')
OUTPUT_DIR = Path('/content/drive/MyDrive/recsys/artifacts/two_tower/baseline_v1')

assert GOLD_ROOT.exists(), f'Missing Gold root: {GOLD_ROOT}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Gold root:', GOLD_ROOT)
print('Output dir:', OUTPUT_DIR)

## Project code

Run this notebook from a checked-out copy of the repository. If the repository is not present in `/content/real-time-adaptive-recsys`, clone or upload it before running the training cell.

In [ ]:
PROJECT_ROOT = Path('/content/real-time-adaptive-recsys')
if not PROJECT_ROOT.exists():
    !git clone https://github.com/kdnehihi/real-time-adaptive-recsys.git /content/real-time-adaptive-recsys
%cd /content/real-time-adaptive-recsys

In [ ]:
!pip install -q torch==2.3.1 pyarrow==15.0.2 pandas==2.2.2 numpy==1.26.4 mlflow==2.14.3

## Write Colab config

MLflow is disabled here; metrics and artifacts are saved directly to Drive.

In [ ]:
import json

config = {
    'run_name': 'two_tower_baseline_colab_v1',
    'gold_root': str(GOLD_ROOT),
    'output_dir': str(OUTPUT_DIR),
    'als_summary_path': '/content/drive/MyDrive/recsys/data/gold/als/v1_colab/evaluation_summary.json',
    'mlflow_tracking_uri': None,
    'mlflow_experiment': 'two_tower_baseline',
    'use_mlflow': False,
    'seed': 42,
    'max_train_examples': 2_000_000,
    'max_validation_examples': 300_000,
    'target_classes': ['STRONG_POSITIVE'],
    'batch_size': 2048,
    'eval_batch_size': 2048,
    'epochs': 3,
    'learning_rate': 0.0003,
    'weight_decay': 0.00001,
    'retrieval_dim': 128,
    'user_hidden_dims': [256, 128],
    'item_hidden_dims': [256, 128],
    'dropout': 0.1,
    'temperature': 0.07,
    'top_ks': [10, 50, 100],
    'eval_max_batches': 100,
    'num_workers': 0,
    'device': 'auto',
    'full_video_vocab_size': 3215508,
    'notes': 'Colab baseline. Uses Gold only, no Silver. Saves artifacts without MLflow.'
}

config_path = Path('configs/two_tower_colab_runtime.json')
config_path.write_text(json.dumps(config, indent=2))
config_path

In [ ]:
!python scripts/train_two_tower_baseline.py --config configs/two_tower_colab_runtime.json

In [ ]:
metrics_path = OUTPUT_DIR / 'metrics.json'
comparison_path = OUTPUT_DIR / 'comparison_summary.json'
print(metrics_path)
print(comparison_path)
print(metrics_path.read_text()[:3000])